# Urdu Voice — Urdu voice-clone test (OmniVoice)

**First:** Runtime → Change runtime type → **T4 GPU** → Save. Then run each cell in order (▶ button).

Only clone **your own** voice.

In [ ]:
# 1) Install (2-4 min)
!pip -q install omnivoice soundfile
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU!")

In [ ]:
# 2) Load model (first time downloads weights)
from omnivoice import OmniVoice
import soundfile as sf, torch
model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)
print("model ready")

## 3) Record your voice
Run the cell, click **Start**, read this sentence clearly (5-10 seconds), click **Stop**:

> **السلام علیکم! آج ہم اردو میں اپنا پہلا وائس اوور بنا رہے ہیں۔**

In [ ]:
REF_TEXT = "السلام علیکم! آج ہم اردو میں اپنا پہلا وائس اوور بنا رہے ہیں۔"
from IPython.display import display, Javascript
from google.colab import output
import base64, subprocess
display(Javascript('''
(async()=>{
 const s=await navigator.mediaDevices.getUserMedia({audio:true});
 const r=new MediaRecorder(s); const ch=[];
 const b=document.createElement('button'); b.textContent='Start'; b.style.fontSize='20px';
 document.querySelector('#output-area').appendChild(b);
 r.ondataavailable=e=>ch.push(e.data);
 r.onstop=async()=>{const blob=new Blob(ch); const rd=new FileReader();
   rd.onloadend=()=>google.colab.kernel.invokeFunction('save_audio',[rd.result.split(',')[1]],{});
   rd.readAsDataURL(blob); b.textContent='Saved ✓';};
 b.onclick=()=>{ if(r.state==='inactive'){r.start(); b.textContent='Stop';} else {r.stop(); s.getTracks().forEach(t=>t.stop());} };
})();'''))
def save_audio(b64):
    open("ref.webm","wb").write(base64.b64decode(b64))
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i","ref.webm","-ac","1","-ar","24000","ref.wav"])
    print("ref.wav saved")
output.register_callback('save_audio', save_audio)

*(Alternative: upload a WAV/MP3 of yourself with the folder icon on the left, then set `!ffmpeg -y -i yourfile.mp3 -ac 1 -ar 24000 ref.wav` in a new cell.)*

In [ ]:
# 4) Listen to your recording
from IPython.display import Audio
Audio("ref.wav")

In [ ]:
# 5) Type any Urdu script and generate in your voice
TEXT = "یہ میری اپنی آواز میں بنایا گیا ایک آزمائشی جملہ ہے۔ کیا یہ بالکل میری طرح لگتا ہے؟"
audio = model.generate(text=TEXT, ref_audio="ref.wav", ref_text=REF_TEXT)
sf.write("out.wav", audio[0], 24000)
Audio("out.wav")

In [ ]:
# 6) Download the result
from google.colab import files
files.download("out.wav")